# Stock Valuation Calculations

### This file runs through the calculation for a DCF stock valuation using unlevered free cash flow

#### (i) loading packages, getting live stock data

In [14]:
##packages
import pandas as pd
import numpy as np
import numpy_financial as fin
import matplotlib as plt
import yfinance as yf

In [15]:
##getting stock info
ticker = yf.Ticker("XRO.AX")

#current/live prices in AUD
info = ticker.info
current_price = info.get("currentPrice")
print(f"Current price: ${current_price}")

Current price: $57.36


In [16]:
## manual inputs (can change values if necessary)

RISK_FREE_RATE = 0.045     ##10yr Aus govt bond yield (approx)
EQUITY_RISK_PREMIUM = 0.055   ##long run ERP estimate
TERMINAL_GROWTH_RATE = 0.02   ## long run AUS GDP growth rate / RBA nflation target

#### Data sources in this notebook

- **Live data sourced from `yfinance`**: current price, shares outstanding, beta,
  total debt, cash, and historical EBIT / D&A / Capex / working capital / tax rate / interest
  expense (used to derive cost of debt).
- **Hardcoded from the FactSet tear sheet (Report as of 21 Sep '26)**: the explicit 2-year
  Free Cash Flow forecast (Mar '27E, Mar '28E). 

In [17]:
## TEAR SHEET DATA (FactSet, Xero Limited (XRO-AU), Report as of 21 Sep '26)
## Hardcoded because these are forward analyst CONSENSUS estimates
## Tear sheet reports these in $ MILLIONS AUD so are multiplied by 1e6 here so they sit in the same raw-dollar units yfinance uses everywhere else in this notebook.
TEAR_SHEET_FCF_FORECAST_AUD = {
    "Mar '27E": 326.92e6,   # A$326.92M
    "Mar '28E": 519.77e6,   # A$519.77M
}

In [18]:
## pull live market data
def get_market_data(ticker_symbol: str) -> dict:
    ticker = yf.Ticker(ticker_symbol)
    info = ticker.info
    #using hist as backup if info is not updated
    hist= ticker.history(period="1d")
    current_price = hist["Close"].iloc[-1] if not hist.empty else info.get("currentPrice")

    return{
        "name": info.get("longName", ticker_symbol),
        "currency": info.get("currency", "AUD"),
        "current_price": current_price,
        "shares_outstanding": info.get("sharesOutstanding"),
        "beta": info.get("beta"),
        "total_debt": info.get("totalDebt", 0) or 0,
        "cash": info.get("totalCash", 0) or 0,
    }

In [19]:
## pulling historical financials (EBIT, D&A, Capex, WC, Interest fr financial statements, rather than hardcoded values)

def get_financials(ticker_symbol: str) -> dict:

    ticker = yf.Ticker(ticker_symbol)
    income_stmt = ticker.financials         # income statement
    balance_sheet = ticker.balance_sheet
    cashflow = ticker.cashflow

    # searching for first matching row name rather than hardcoding an exact label
    def find_row(df: pd.DataFrame, candidates: list[str]) -> pd.Series:
        for name in candidates:
            if name in df.index:
                return df.loc[name]
        raise KeyError(f"None of {candidates} found in statement rows: {list(df.index)}")

    ebit = find_row(income_stmt, ["EBIT", "Operating Income"])
    pretax_income = find_row(income_stmt, ["Pretax Income", "Income Before Tax"])
    tax_expense = find_row(income_stmt, ["Tax Provision", "Income Tax Expense"])
    interest_expense = find_row(income_stmt, ["Interest Expense"])

    # D&A: try the cash flow statement first (label varies by ticker), and if
    # none of those hit, derive it as EBITDA - EBIT from the income statement
    # instead of hardcoding 0 - a 0 strips out a real, often large, non-cash
    # add-back and badly understates FCF.
    try:
        depreciation_amortisation = find_row(
            cashflow,
            [
                "Depreciation And Amortization",
                "Depreciation Amortization Depletion",
                "Depreciation",
            ],
        )
    except KeyError:
        ebitda = find_row(income_stmt, ["EBITDA", "Normalized EBITDA"])
        depreciation_amortisation = (ebitda - ebit).reindex(ebit.index)

    capex = find_row(cashflow, ["Capital Expenditure", "Purchase Of PPE"])

    current_assets = find_row(balance_sheet, ["Current Assets", "Total Current Assets"])
    current_liabilities = find_row(balance_sheet, ["Current Liabilities", "Total Current Liabilities"])
    working_capital = current_assets - current_liabilities

    # Effective tax rate, derived (not hardcoded) from the last available year
    effective_tax_rate = float(tax_expense.iloc[0] / pretax_income.iloc[0])

    # Cost of debt, derived from interest expense / total debt (kd)
    total_debt_bs = find_row(balance_sheet, ["Total Debt"])
    cost_of_debt = float(interest_expense.iloc[0] / total_debt_bs.iloc[0])

    return {
        "ebit": ebit,                                   # most recent year first
        "da": depreciation_amortisation,
        "capex": capex,
        "working_capital": working_capital,
        "effective_tax_rate": effective_tax_rate,
        "cost_of_debt": abs(cost_of_debt),
    }

### 1. Free Cash Flow
$$
\text{Unlevered FCF} = \text{EBIT}(1-\text{tax rate}) + \text{Depreciation and Ammortisation} - \Delta \text{WC} - \text{Capex}
$$

In [20]:
def calculate_historical_ufcf(fin: dict) -> pd.Series:
    ebit = fin["ebit"]
    da = fin["da"].reindex(ebit.index)
    capex = fin["capex"].reindex(ebit.index)
    wc = fin["working_capital"].reindex(ebit.index)

    tax = fin["effective_tax_rate"]

    # keep the SIGN here: a rise in working capital consumes cash (subtract
    # it), a fall frees up cash (adding it back increases FCF). Wrapping this
    # in .abs() destroyed that directional information and was corrupting FCF.
    change_in_wc = wc.diff(-1)  # current year minus prior year, since cols run newest -> oldest

    # Capex is a cash outflow either way, so .abs() here is correct - unlike
    # working capital, its sign carries no extra information we need to keep.
    ufcf = ebit * (1 - tax) + da - change_in_wc - capex.abs()
    return ufcf.dropna().sort_index()

### 2. Terminal value
$$
\text{TV} = \frac{\text{FCF}_\text{N+1}}{\text{WACC} - \text{g}}
$$

##### perpetual growth model represents the estimated value of business beyond an explicit fcst period

In [21]:
##calculating perpetual growth TV model
def calculate_terminal_value(final_year_fcf: float, wacc: float, g: float) -> float:
    fcf_next = final_year_fcf * (1 + g)
    return fcf_next / (wacc - g)

### 3. Discounting Back to PV
#### the terminal value represents the stock value at the end of the forecast period, to find its contribution to current value, must discount back to present using discount rate (WACC)

### 3.1 Calculating WACC
$$
\text{WACC} = \text{w}_d \text{k}_d (1- \text{T}) + \text{w}_e \text{k}_e
$$

##### Where T = corporate tax rate (20%)

In [22]:
##process: get cash flows > discount them > get EV > Get equity value > get price per share

def calculate_wacc(market: dict, fin: dict) -> tuple[float, float]:
    """LIVE / DYNAMIC (yfinance) - calculates WACC and cost of equity (ke).
    Touches nothing about FCF or valuation - just the discount rate."""
    cost_of_equity = RISK_FREE_RATE + market["beta"] * EQUITY_RISK_PREMIUM
    equity_value = market["current_price"] * market["shares_outstanding"]
    debt_value = market["total_debt"]
    total_capital = equity_value + debt_value

    wacc = (
        equity_value / total_capital * cost_of_equity
        + debt_value / total_capital * fin["cost_of_debt"] * (1 - fin["effective_tax_rate"])
    )
    return wacc, cost_of_equity

In [23]:
## discounting forecast FCF and terminal value to today
## the forecast FCF is from tear sheet (hardcoded) The WACC is live from yfinance. 
## Terminal value is computed off the last forecase yr. discounts each fcst yr FCF and TV back to PV and prints ea year
def project_fcf_and_terminal_value(projected_fcf: list[float], wacc: float, g: float) -> dict:

    forecast_years = len(projected_fcf)
    terminal_value = calculate_terminal_value(projected_fcf[-1], wacc, g)

    print("Forecast year cash flows:")
    discounted_fcfs = []
    for year, fcf in enumerate(projected_fcf, start=1):
        discounted = fcf / (1 + wacc) ** year
        discounted_fcfs.append(discounted)
        print(f"  Year {year}: FCF = ${fcf:,.0f}   ->   PV = ${discounted:,.0f}")

    discounted_terminal_value = terminal_value / (1 + wacc) ** forecast_years
    print(f"Terminal value (undiscounted): ${terminal_value:,.0f}")
    print(f"Terminal value (discounted):   ${discounted_terminal_value:,.0f}")

    return {
        "forecast_years": forecast_years,
        "discounted_fcfs": discounted_fcfs,
        "terminal_value": terminal_value,
        "discounted_terminal_value": discounted_terminal_value,
    }


In [24]:
## Calculating enterprise calue, equity value, net debt, then price per share. 
def calculate_valuation(
    discounted_fcfs: list[float],
    discounted_terminal_value: float,
    total_debt: float,
    cash: float,
    shares_outstanding: float,
) -> dict:

    enterprise_value = sum(discounted_fcfs) + discounted_terminal_value

    # EV = Market Cap + Net Debt  ->  Market Cap = EV - Net Debt
    # total_debt / cash here stay fully dynamic (yfinance)
    net_debt = total_debt - cash
    equity_value = enterprise_value - net_debt
    implied_price_per_share = equity_value / shares_outstanding

    return {
        "enterprise_value": enterprise_value,
        "net_debt": net_debt,
        "equity_value": equity_value,
        "implied_price_per_share": implied_price_per_share,
    }

In [25]:
def run_dcf(ticker_symbol: str = "XRO.AX"):
    # LIVE data
    market = get_market_data(ticker_symbol)
    fin = get_financials(ticker_symbol)
    historical_ufcf = calculate_historical_ufcf(fin)

    # HARDCODED (FactSet tear sheet): explicit forecast years and values come
    # directly from analyst consensus estimates
    projected_fcf = list(TEAR_SHEET_FCF_FORECAST_AUD.values())

    wacc, ke = calculate_wacc(market, fin)
    fcf_result = project_fcf_and_terminal_value(projected_fcf, wacc, TERMINAL_GROWTH_RATE)
    valuation = calculate_valuation(
        fcf_result["discounted_fcfs"],
        fcf_result["discounted_terminal_value"],
        market["total_debt"],
        market["cash"],
        market["shares_outstanding"],
    )

    return {
        "company": market["name"],
        "current_price": market["current_price"],
        "cost_of_equity_ke": ke,
        "wacc": wacc,
        "historical_ufcf": historical_ufcf,
        "projected_fcf": projected_fcf,
        "forecast_years": fcf_result["forecast_years"],
        "terminal_value": fcf_result["terminal_value"],
        **valuation,
    }

In [26]:
if __name__ == "__main__":
    ticker_symbol = "XRO.AX"
    result = run_dcf(ticker_symbol)

    print(f"\nDCF Valuation: {result['company']} ({ticker_symbol})")
    print("=" * 50)
    print("[dynamic - yfinance]")
    print(f"Cost of equity (ke):        {result['cost_of_equity_ke']:.2%}")
    print(f"WACC:                       {result['wacc']:.2%}")
    print(f"Current share price:        ${result['current_price']:.2f}")
    print(f"Net debt:                   ${result['net_debt']:,.0f}")
    print()
    print(f"[hardcoded - FactSet tear sheet, {result['forecast_years']}-year explicit forecast]")
    for label, value in zip(TEAR_SHEET_FCF_FORECAST_AUD.keys(), result['projected_fcf']):
        print(f"  {label} FCF:              ${value:,.0f}")
    print()
    print("[calculated from the above]")
    print(f"Terminal value:              ${result['terminal_value']:,.0f}")
    print(f"Enterprise value:            ${result['enterprise_value']:,.0f}")
    print(f"Equity value:                ${result['equity_value']:,.0f}")
    print("-" * 50)
    print(f"DCF-implied share price:    ${result['implied_price_per_share']:.2f}")

## Note: D&A is sourced from the cash flow statement (with an EBITDA - EBIT fallback)
## The FCF forecast used for the terminal value is the FactSet tear sheet's analyst consensus (Mar '27E, Mar '28E)

Forecast year cash flows:
  Year 1: FCF = $326,920,000   ->   PV = $303,502,301
  Year 2: FCF = $519,770,000   ->   PV = $447,973,378
Terminal value (undiscounted): $9,275,400,126
Terminal value (discounted):   $7,994,174,981

DCF Valuation: Xero Limited (XRO.AX)
[dynamic - yfinance]
Cost of equity (ke):        8.53%
WACC:                       7.72%
Current share price:        $57.36
Net debt:                   $190,845,056

[hardcoded - FactSet tear sheet, 2-year explicit forecast]
  Mar '27E FCF:              $326,920,000
  Mar '28E FCF:              $519,770,000

[calculated from the above]
Terminal value:              $9,275,400,126
Enterprise value:            $8,745,650,661
Equity value:                $8,554,805,605
--------------------------------------------------
DCF-implied share price:    $48.42


#### Conclusion: investment conditional, under 2 y fcst XRO is valued below current price.